# 04. Regresión Softmax: Clasificación Multi-Clase

**Nivel:** 🟡 Intermedio  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** [03. Regresión Logística](03-regresion-logistica.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Extender clasificación binaria a múltiples clases
- Entender y aplicar la función softmax
- Implementar one-hot encoding de etiquetas
- Calcular Categorical Cross-Entropy loss
- Entrenar clasificadores multi-clase desde cero
- Interpretar vectores de probabilidades por clase

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris, load_digits
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades personalizadas
import sys
sys.path.append('../../shared/utils')
from visualization import plot_confusion_matrix, plot_multiclass_probabilities
from testing import test_exercise, check_shape, check_close
from datasets import load_dataset

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

---
## 📌 1. Motivación: Del Mundo Binario al Mundo Multi-Clase

### El Problema del Mundo Real

Trabajas en un sistema de reconocimiento de dígitos escritos a mano para automatizar el procesamiento de cheques bancarios. El sistema debe reconocer los dígitos del 0 al 9.

**Esto NO es clasificación binaria (2 clases).** Necesitas distinguir entre **10 clases diferentes**.

### ¿Por qué no 10 clasificadores binarios?

Podrías entrenar:
- Clasificador 1: ¿Es un 0? (sí/no)
- Clasificador 2: ¿Es un 1? (sí/no)
- ...
- Clasificador 10: ¿Es un 9? (sí/no)

**Problemas:**
- 10 modelos separados (ineficiente)
- Las probabilidades no suman 1 (no son mutuamente excluyentes)
- Puede que varios digan "sí" o ninguno diga "sí"

### La Solución: Softmax

**Softmax** generaliza la sigmoide para K clases:
- Un solo modelo
- Produce K probabilidades que suman exactamente 1
- Interpretación clara: distribución de probabilidad sobre clases

### Aplicaciones Reales

- 🔢 Reconocimiento de dígitos (0-9)
- 🌸 Clasificación de especies (Iris: setosa, versicolor, virginica)
- 📰 Categorización de noticias (deportes, política, tecnología, etc.)
- 🎭 Reconocimiento de emociones (feliz, triste, enojado, neutral, etc.)
- 🗣️ Procesamiento de lenguaje natural (predicción de siguiente palabra)

### La Pregunta Guía

> **¿Cómo extender la clasificación binaria a múltiples clases de forma que las probabilidades predichas sean consistentes y entrenables?**

---
## 📊 2. Intuición Visual: De Sigmoide a Softmax

In [ ]:
# Función softmax
def softmax(z):
    """
    Función softmax con estabilidad numérica.
    
    softmax(z_i) = exp(z_i) / Σ exp(z_j)
    """
    # Restar el máximo para estabilidad numérica (evitar overflow)
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# Ejemplo: 3 clases
logits = np.array([2.0, 1.0, 0.1])  # Salidas lineales (logits)
probs = softmax(logits)

print("Ejemplo de Softmax para 3 clases:")
print(f"\nLogits (salidas lineales): {logits}")
print(f"Probabilidades softmax:     {probs}")
print(f"Suma de probabilidades:     {np.sum(probs):.10f}")
print(f"\n💡 La clase con mayor logit (2.0) tiene la mayor probabilidad ({probs[0]:.3f})")

# Visualización
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Logits (valores brutos)', 'Probabilidades Softmax')
)

# Logits
fig.add_trace(
    go.Bar(x=['Clase 0', 'Clase 1', 'Clase 2'], y=logits, name='Logits',
           marker_color='lightblue'),
    row=1, col=1
)

# Probabilidades
fig.add_trace(
    go.Bar(x=['Clase 0', 'Clase 1', 'Clase 2'], y=probs, name='Probabilidades',
           marker_color='lightgreen'),
    row=1, col=2
)

fig.update_layout(
    title="Transformación Softmax",
    template="plotly_white",
    showlegend=False,
    height=400
)

fig.show()

print("\n💡 Propiedades clave de Softmax:")
print("   • Todas las probabilidades están entre 0 y 1")
print("   • La suma de probabilidades es exactamente 1")
print("   • Amplifica diferencias (efecto 'competencia')")
print("   • Diferenciable (necesario para backpropagation)")

In [ ]:
# Visualización del dataset Iris (3 clases)
iris = load_iris()
X_iris = iris.data[:, :2]  # Usar solo 2 features para visualización
y_iris = iris.target
target_names = iris.target_names

# Crear DataFrame para plotly
df_iris = pd.DataFrame({
    'Sepal Length': X_iris[:, 0],
    'Sepal Width': X_iris[:, 1],
    'Species': [target_names[i] for i in y_iris]
})

fig = px.scatter(
    df_iris,
    x='Sepal Length',
    y='Sepal Width',
    color='Species',
    title="Dataset Iris: 3 Especies de Flores",
    color_discrete_sequence=['red', 'blue', 'green']
)

fig.update_layout(
    template="plotly_white",
    font=dict(size=12),
    height=500
)

fig.show()

print("\n💡 Problema: Dado sepal length y width, predecir la especie (3 clases).")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Notación

| Símbolo | Significado |
|---------|-------------|
| $K$ | Número de clases |
| $y \in \{0, 1, ..., K-1\}$ | Etiqueta de clase (escalar) |
| $\mathbf{y} \in \{0,1\}^K$ | Etiqueta one-hot encoded (vector) |
| $\mathbf{z} \in \mathbb{R}^K$ | Logits (salidas lineales) |
| $\hat{\mathbf{y}} \in [0,1]^K$ | Probabilidades predichas |
| $W \in \mathbb{R}^{n \times K}$ | Matriz de pesos |
| $\mathbf{b} \in \mathbb{R}^K$ | Vector de bias |

### El Modelo Softmax

**Paso 1: Combinaciones lineales** (una por clase)

$$
\mathbf{z} = W^T \mathbf{x} + \mathbf{b} \quad \text{donde } \mathbf{z} = [z_0, z_1, ..., z_{K-1}]^T \tag{1}
$$

Para cada clase $k$:
$$
z_k = \mathbf{w}_k^T \mathbf{x} + b_k \tag{2}
$$

**Paso 2: Aplicar softmax**

$$
\hat{y}_k = \text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}} \tag{3}
$$

**Interpretación:** $\hat{y}_k = P(y=k | \mathbf{x})$ es la probabilidad de que el ejemplo pertenezca a la clase $k$.

**Propiedades:**
1. $\hat{y}_k \in (0, 1)$ para todo $k$
2. $\sum_{k=0}^{K-1} \hat{y}_k = 1$
3. $\arg\max_k \hat{y}_k = \arg\max_k z_k$ (la clase predicha es la de mayor logit)

### One-Hot Encoding

Para entrenar, convertimos las etiquetas escalares a vectores:

$$
y = 2 \quad \rightarrow \quad \mathbf{y} = [0, 0, 1, 0, ..., 0]^T \tag{4}
$$

El vector tiene un 1 en la posición de la clase verdadera y 0s en el resto.

### Función de Pérdida: Categorical Cross-Entropy

Para un solo ejemplo:

$$
\mathcal{L}(\mathbf{y}, \hat{\mathbf{y}}) = -\sum_{k=0}^{K-1} y_k \log(\hat{y}_k) \tag{5}
$$

Como $\mathbf{y}$ es one-hot, solo el término de la clase verdadera contribuye:

$$
\mathcal{L} = -\log(\hat{y}_{\text{true}}) \tag{6}
$$

Para todo el dataset:

$$
J(W, \mathbf{b}) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=0}^{K-1} y_k^{(i)} \log(\hat{y}_k^{(i)}) \tag{7}
$$

### Gradientes

La derivada respecto a los logits es elegante:

$$
\frac{\partial \mathcal{L}}{\partial z_k} = \hat{y}_k - y_k \tag{8}
$$

Para actualizar los pesos de la clase $k$:

$$
\frac{\partial J}{\partial \mathbf{w}_k} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_k^{(i)} - y_k^{(i)}) \mathbf{x}^{(i)} \tag{9}
$$

$$
\frac{\partial J}{\partial b_k} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_k^{(i)} - y_k^{(i)}) \tag{10}
$$

### Ejemplo Numérico

Clasificación de flores con 3 clases:

**Features de una flor:** $\mathbf{x} = [5.1, 3.5]$ (sepal length, width)

**Pesos (simplificados):**
$$
W = \begin{bmatrix}
0.5 & -0.2 & 0.1 \\
0.3 & 0.8 & -0.5
\end{bmatrix}, \quad
\mathbf{b} = \begin{bmatrix}
-1.0 \\
0.5 \\
0.2
\end{bmatrix}
$$

**Cálculo:**

$$
\begin{align}
z_0 &= 0.5(5.1) + 0.3(3.5) - 1.0 = 1.60 \\
z_1 &= -0.2(5.1) + 0.8(3.5) + 0.5 = 2.28 \\
z_2 &= 0.1(5.1) - 0.5(3.5) + 0.2 = -1.04
\end{align}
$$

$$
\begin{align}
\text{Suma exp: } & e^{1.60} + e^{2.28} + e^{-1.04} = 4.95 + 9.77 + 0.35 = 15.07 \\
\hat{y}_0 &= \frac{4.95}{15.07} = 0.328 \\
\hat{y}_1 &= \frac{9.77}{15.07} = 0.648 \\
\hat{y}_2 &= \frac{0.35}{15.07} = 0.023
\end{align}
$$

**Predicción:** Clase 1 (mayor probabilidad: 64.8%)

In [ ]:
# Verificación del ejemplo numérico
x = np.array([5.1, 3.5])
W = np.array([
    [0.5, -0.2, 0.1],
    [0.3, 0.8, -0.5]
])
b = np.array([-1.0, 0.5, 0.2])

# Cálculo
z = W.T @ x + b
probs = softmax(z)

print("Verificación del Ejemplo Numérico:")
print(f"\nLogits z: {z}")
print(f"Probabilidades: {probs}")
print(f"Suma: {np.sum(probs):.10f}")
print(f"\nClase predicha: {np.argmax(probs)} (probabilidad: {np.max(probs):.3f})")

### Comparación: Sigmoide vs Softmax

| Aspecto | Sigmoide (Binaria) | Softmax (Multi-clase) |
|---------|-------------------|----------------------|
| **Número de clases** | 2 | K (cualquiera) |
| **Salidas** | 1 probabilidad | K probabilidades |
| **Suma de salidas** | N/A (se infiere 1-p) | Exactamente 1 |
| **Fórmula** | $\frac{1}{1+e^{-z}}$ | $\frac{e^{z_k}}{\sum e^{z_j}}$ |
| **Loss** | Binary Cross-Entropy | Categorical Cross-Entropy |
| **Caso especial** | Softmax con K=2 | Generaliza sigmoide |

---
## 💻 4. Implementación Desde Cero

In [ ]:
class RegresionSoftmax:
    """
    Implementación desde cero de Regresión Softmax para clasificación multi-clase.
    
    Parameters:
    -----------
    learning_rate : float
        Tasa de aprendizaje
    n_iterations : int
        Número de iteraciones de entrenamiento
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iter = n_iterations
        
        # Parámetros del modelo
        self.W = None  # Matriz de pesos (n_features, n_classes)
        self.b = None  # Vector de bias (n_classes,)
        
        # Tracking
        self.losses = []
        self.n_classes = None
    
    @staticmethod
    def softmax(z):
        """
        Función softmax con estabilidad numérica.
        
        Parameters:
        -----------
        z : np.ndarray, shape (n_samples, n_classes)
            Logits
        
        Returns:
        --------
        probs : np.ndarray, shape (n_samples, n_classes)
            Probabilidades
        """
        # Restar max para estabilidad (evitar overflow)
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    @staticmethod
    def one_hot_encode(y, n_classes):
        """
        Convierte etiquetas a representación one-hot.
        
        Ejemplo: y=2, n_classes=4 → [0, 0, 1, 0]
        
        Parameters:
        -----------
        y : np.ndarray, shape (n_samples,)
            Etiquetas (enteros de 0 a n_classes-1)
        n_classes : int
            Número de clases
        
        Returns:
        --------
        y_onehot : np.ndarray, shape (n_samples, n_classes)
            Matriz one-hot
        """
        n_samples = len(y)
        y_onehot = np.zeros((n_samples, n_classes))
        y_onehot[np.arange(n_samples), y] = 1
        return y_onehot
    
    def categorical_cross_entropy(self, y_true, y_pred):
        """
        Calcula Categorical Cross-Entropy Loss.
        
        L = -1/m * Σ Σ y_true[k] * log(y_pred[k])
        """
        # Clip para evitar log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        # Solo el término de la clase verdadera contribuye
        loss = -np.mean(np.sum(y_true * np.log(y_pred), axis=1))
        return loss
    
    def fit(self, X, y):
        """
        Entrena el modelo usando gradient descent.
        
        Parameters:
        -----------
        X : np.ndarray, shape (n_samples, n_features)
            Features de entrenamiento
        y : np.ndarray, shape (n_samples,)
            Etiquetas (enteros de 0 a n_classes-1)
        """
        n_samples, n_features = X.shape
        self.n_classes = len(np.unique(y))
        
        # Inicialización de parámetros (Xavier/Glorot)
        self.W = np.random.randn(n_features, self.n_classes) * 0.01
        self.b = np.zeros(self.n_classes)
        
        # One-hot encode de etiquetas
        y_onehot = self.one_hot_encode(y, self.n_classes)
        
        print(f"🏃 Iniciando entrenamiento...")
        print(f"   Clases: {self.n_classes}")
        print(f"   Ejemplos: {n_samples}")
        print(f"   Features: {n_features}\n")
        
        # Gradient descent
        for i in range(self.n_iter):
            # Forward pass
            # 1. Calcular logits: z = X * W + b
            z = X @ self.W + self.b  # (n_samples, n_classes)
            
            # 2. Aplicar softmax
            y_pred = self.softmax(z)  # (n_samples, n_classes)
            
            # 3. Calcular pérdida
            loss = self.categorical_cross_entropy(y_onehot, y_pred)
            self.losses.append(loss)
            
            # Backward pass: calcular gradientes
            # Error: (y_pred - y_true) para cada clase
            error = y_pred - y_onehot  # (n_samples, n_classes)
            
            # Gradientes
            dW = (1 / n_samples) * (X.T @ error)  # (n_features, n_classes)
            db = (1 / n_samples) * np.sum(error, axis=0)  # (n_classes,)
            
            # Actualizar parámetros
            self.W -= self.lr * dW
            self.b -= self.lr * db
            
            # Logging
            if i % max(1, self.n_iter // 10) == 0:
                # Calcular accuracy
                y_pred_class = np.argmax(y_pred, axis=1)
                accuracy = np.mean(y_pred_class == y)
                print(f"Iter {i:4d} - Loss: {loss:.4f} - Accuracy: {accuracy:.4f}")
        
        print(f"\n✅ Entrenamiento completado")
        print(f"   Loss final: {self.losses[-1]:.4f}")
        
        return self
    
    def predict_proba(self, X):
        """
        Predice probabilidades para cada clase.
        
        Returns:
        --------
        probabilities : np.ndarray, shape (n_samples, n_classes)
            Matriz de probabilidades
        """
        z = X @ self.W + self.b
        return self.softmax(z)
    
    def predict(self, X):
        """
        Predice la clase más probable.
        
        Returns:
        --------
        predictions : np.ndarray, shape (n_samples,)
            Clase predicha para cada ejemplo
        """
        probabilities = self.predict_proba(X)
        return np.argmax(probabilities, axis=1)
    
    def score(self, X, y):
        """Calcula accuracy"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

print("✅ Clase RegresionSoftmax definida")

### Probemos con el dataset Iris (3 clases)

In [ ]:
# Cargar datos
iris = load_iris()
X, y = iris.data, iris.target

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"📊 Dataset Iris:")
print(f"   Clases: {len(np.unique(y))} ({iris.target_names})")
print(f"   Entrenamiento: {X_train.shape[0]} ejemplos")
print(f"   Prueba: {X_test.shape[0]} ejemplos")
print(f"   Features: {X_train.shape[1]}")

In [ ]:
# Entrenar modelo
modelo = RegresionSoftmax(learning_rate=0.1, n_iterations=1000)
modelo.fit(X_train, y_train)

In [ ]:
# Visualizar convergencia
fig = go.Figure()
fig.add_trace(go.Scatter(y=modelo.losses, mode='lines', name='Loss'))
fig.update_layout(
    title="Convergencia: Categorical Cross-Entropy",
    xaxis_title="Iteración",
    yaxis_title="Loss",
    template="plotly_white"
)
fig.show()

In [ ]:
# Evaluar modelo
y_pred_test = modelo.predict(X_test)
y_proba_test = modelo.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred_test)
print(f"📊 Accuracy en test: {accuracy:.4f}")

# Reporte de clasificación
print("\n" + classification_report(y_test, y_pred_test, target_names=iris.target_names))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_test)

fig = px.imshow(
    cm,
    labels=dict(x="Predicho", y="Real", color="Cantidad"),
    x=iris.target_names,
    y=iris.target_names,
    text_auto=True,
    color_continuous_scale='Blues'
)
fig.update_layout(title="Matriz de Confusión - Iris", template="plotly_white")
fig.show()

print("\n💡 Diagonal: Predicciones correctas")
print("   Fuera de diagonal: Confusiones entre clases")

In [ ]:
# Visualizar probabilidades para algunos ejemplos
n_examples = 5
example_probs = y_proba_test[:n_examples]
example_true = y_test[:n_examples]
example_pred = y_pred_test[:n_examples]

fig = go.Figure()

for i in range(n_examples):
    fig.add_trace(go.Bar(
        x=iris.target_names,
        y=example_probs[i],
        name=f'Ej {i+1}: Real={iris.target_names[example_true[i]]}, Pred={iris.target_names[example_pred[i]]}'
    ))

fig.update_layout(
    title="Distribución de Probabilidades para Ejemplos de Test",
    xaxis_title="Clase",
    yaxis_title="Probabilidad",
    template="plotly_white",
    barmode='group',
    height=500
)

fig.show()

print("\n💡 Nota cómo las probabilidades suman 1 para cada ejemplo.")
print("   La clase con mayor probabilidad es la predicha.")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Scikit-learn automáticamente usa softmax para multi-clase
sklearn_model = LogisticRegression(multi_class='multinomial', max_iter=1000)
sklearn_model.fit(X_train, y_train)

# Predicciones
y_pred_sklearn = sklearn_model.predict(X_test)
y_proba_sklearn = sklearn_model.predict_proba(X_test)

# Comparación
accuracy_ours = accuracy_score(y_test, y_pred_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("📊 Comparación: Nuestra Implementación vs Scikit-learn\n")
print("="*50)
print(f"{'Métrica':<20} {'Nuestra':<15} {'Scikit-learn':<15}")
print("="*50)
print(f"{'Accuracy':<20} {accuracy_ours:<15.4f} {accuracy_sklearn:<15.4f}")
print("="*50)

# Comparar probabilidades de un ejemplo
idx = 0
print(f"\n💡 Probabilidades para ejemplo {idx}:")
print(f"   Nuestra:      {y_proba_test[idx]}")
print(f"   Scikit-learn: {y_proba_sklearn[idx]}")
print(f"   Clase real:   {iris.target_names[y_test[idx]]}")

print("\n✅ Resultados muy similares!")
print("\n💡 Ventajas de Scikit-learn:")
print("   • Estrategias one-vs-rest y multinomial")
print("   • Regularización L1/L2")
print("   • Optimizadores avanzados (lbfgs, saga)")
print("   • Manejo automático de multi-clase")

---
## 🎯 6. Ejercicios Prácticos

### 🟢 Ejercicio 1: Clasificación de Dígitos (10 clases)

Usa el dataset de dígitos (0-9) para entrenar un clasificador de 10 clases.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Clasificar dígitos escritos a mano (0-9)
    
    Instrucciones:
    1. Carga el dataset 'digits' con load_digits()
    2. Normaliza con StandardScaler
    3. Entrena un modelo RegresionSoftmax
    4. Calcula accuracy y muestra confusion matrix
    
    Returns:
    --------
    accuracy : float
    """
    # TODO: Tu código aquí
    # from sklearn.datasets import load_digits
    # digits = load_digits()
    # ...
    
    pass

# Descomentar para probar
# acc = ejercicio_1()
# print(f"\n📊 Accuracy obtenida: {acc:.4f}")

### 🟡 Ejercicio 2: Análisis de Confianza

Analiza cómo la confianza del modelo se relaciona con la precisión.

In [ ]:
def ejercicio_2():
    """
    Objetivo: Analizar la relación entre confianza y acierto
    
    Instrucciones:
    1. Entrena un modelo de clasificación
    2. Para cada predicción en test, calcula:
       - Confianza: max(probabilidades)
       - Correcto: 1 si predicción == real, 0 si no
    3. Divide en bins de confianza [0-0.3, 0.3-0.6, 0.6-0.8, 0.8-1.0]
    4. Calcula accuracy en cada bin
    5. Visualiza: confianza vs accuracy
    
    Hipótesis: A mayor confianza, mayor accuracy
    
    Returns:
    --------
    dict : {bin_range: accuracy}
    """
    # TODO: Tu código aquí
    # confidence = np.max(y_proba, axis=1)
    # correct = (y_pred == y_test).astype(int)
    
    pass

# Descomentar para probar
# results = ejercicio_2()
# print("\n📊 Accuracy por nivel de confianza:")
# for bin_range, acc in results.items():
#     print(f"   {bin_range}: {acc:.3f}")

### 🔴 Ejercicio 3: Calibración de Probabilidades

Las probabilidades de softmax pueden estar "mal calibradas". Implementa temperature scaling.

In [ ]:
def ejercicio_3():
    """
    Objetivo: Calibrar probabilidades con temperature scaling
    
    Instrucciones:
    1. Temperature scaling modifica softmax:
       softmax(z/T) donde T es la "temperatura"
    2. T > 1: probabilidades más uniformes (menos confiadas)
       T < 1: probabilidades más extremas (más confiadas)
       T = 1: softmax normal
    3. Implementa una función softmax_with_temperature(z, T)
    4. Prueba con T = [0.5, 1.0, 2.0, 5.0]
    5. Visualiza cómo cambian las distribuciones
    
    Usa Expected Calibration Error (ECE) para evaluar:
    ECE mide la diferencia entre confianza y accuracy real
    
    Returns:
    --------
    dict : {temperature: ECE}
    """
    # TODO: Tu código aquí
    # def softmax_with_temperature(z, T):
    #     return softmax(z / T)
    
    pass

# Descomentar para probar
# results = ejercicio_3()
# print("\n📊 ECE por temperatura:")
# for T, ece in results.items():
#     print(f"   T={T}: ECE={ece:.4f}")

---
## 📚 7. Resumen y Recursos

### 🎯 Puntos Clave

1. **Softmax** generaliza la sigmoide para K clases: $\hat{y}_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$

2. **One-hot encoding** convierte etiquetas escalares en vectores: $y=2 \rightarrow [0,0,1,0,...]$

3. **Categorical Cross-Entropy** es la pérdida apropiada: $\mathcal{L} = -\sum_k y_k \log(\hat{y}_k)$

4. **Propiedades de softmax**:
   - Todas las probabilidades entre 0 y 1
   - Suman exactamente 1
   - Diferenciable (esencial para backprop)

5. **Implementación**: Requiere matriz de pesos W (n_features × n_classes) en lugar de vector

6. **Es la última capa de redes neuronales** para clasificación multi-clase

### 🔗 Recursos Adicionales

#### 📄 Papers Fundamentales

- **"A Model of Classification"** - Luce (1959)
  - Introduce el modelo de elección softmax en psicología matemática

- **"On Calibration of Modern Neural Networks"** - Guo et al. (2017)
  - [Link](https://arxiv.org/abs/1706.04599)
  - Explica por qué las probabilidades pueden estar mal calibradas

#### 📖 Recursos Educativos

- **Deep Learning Book** - Goodfellow, Bengio, Courville
  - Capítulo 6.2.2.3: Softmax Units for Multinoulli Output

- **CS231n - Linear Classification**
  - [http://cs231n.github.io/linear-classify/](http://cs231n.github.io/linear-classify/)

#### 💻 Implementaciones

- [NumPy: Softmax numerically stable](https://stackoverflow.com/questions/34968722/how-to-implement-the-softmax-function-in-python)
- [TensorFlow: tf.nn.softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax)
- [PyTorch: torch.nn.functional.softmax](https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html)

### 🤔 Preguntas para Reflexionar

1. **¿Por qué softmax y no simplemente normalizar los logits?**
   - La exponencial amplifica diferencias
   - Tiene propiedades matemáticas convenientes para gradientes
   - Conexión con modelos probabilísticos (MaxEnt)

2. **¿Cuándo usar multinomial vs one-vs-rest?**
   - **Multinomial (softmax)**: Clases mutuamente excluyentes
   - **One-vs-rest**: Cuando un ejemplo puede pertenecer a múltiples clases

3. **¿Qué hacer con clases desbalanceadas en multi-clase?**
   - Class weights en la pérdida
   - Oversampling/undersampling
   - Focal Loss (usado en detección de objetos)

### 📊 Tabla de Referencia: Funciones de Activación Final

| Tarea | Función | Output | Loss |
|-------|---------|--------|------|
| **Regresión** | Lineal (ninguna) | $\mathbb{R}$ | MSE |
| **Clasificación binaria** | Sigmoide | $(0, 1)$ | Binary Cross-Entropy |
| **Clasificación multi-clase** | Softmax | $(0,1)^K$, suma=1 | Categorical Cross-Entropy |
| **Multi-label** | Sigmoide (múltiple) | $[0,1]^K$ | Binary Cross-Entropy |

---

## ➡️ Próximo Paso

En el siguiente notebook, **05. Árboles de Decisión**, cambiaremos completamente de paradigma:

- De modelos **lineales** a modelos **basados en reglas**
- Entender entropía y ganancia de información
- Criterios de splitting (Gini, Entropy)
- Ventajas: interpretabilidad, no requiere normalización
- Desventajas: overfitting, inestabilidad

---

<div align="center">

**🎉 ¡Has completado los fundamentos de clasificación! 🎉**

**Continúa con: [05. Árboles de Decisión](05-arboles-decision.ipynb)**

[← 03. Regresión Logística](03-regresion-logistica.ipynb) | [Índice de ML Clásico](README.md) | [05. Árboles de Decisión →](05-arboles-decision.ipynb)

</div>